In [ ]:
%%writefile ambiente.h
// Contiene: i parametri, gli ostacoli (ostacoli e fori), la maschera della griglia,
// le sorgenti della passata 1 (bordo degli ostacoli)
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>
#include <algorithm>

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3)
struct Punto   { double x, y, z; };
struct Zona { double x0, x1, y0, y1, z0, z1; };       // parallelepipedo: x0 <= x <= x1, y0 <= y <= y1, z0 <= z <= z1


// AMBIENTE: per cambiarlo si modifica solo questa parte
// =====================================================================
// Stanza = cubo [-1, 1]^3, z verticale (z = -1 pavimento). Fuori dal cubo = parete (INF).
const double MARGINE = 0.06;                             // distanza minima dagli ostacoli (raggio del drone + sicurezza)
const double D_SICUREZZA = 0.3;                          // distanza dagli ostacoli oltre la quale si va a velocita' piena
const double V_MIN = 0.1;                                // velocita' minima, a ridosso degli ostacoli (> 0)

const Punto A = { -0.8, -0.6, -0.6 };                    // partenza del drone: prima meta' della stanza, in basso
const Punto B = {  0.85, -0.4, -0.7 };                   // arrivo (sorgente del FIM): dietro l'armadio

const std::vector<Zona> OSTACOLI = {                   // {x0, x1, y0, y1, z0, z1}
    { -0.05, 0.05, -1.0,  1.0, -1.0,  1.0 },             // muro a x = 0: divide la stanza in due
    {  0.35, 0.65, -0.7, -0.1, -1.0, -0.2 },             // armadio appoggiato a terra, oltre il muro
};
const std::vector<Zona> FORI = {                      // zone in cui l'ostacolo viene tolto
    { -0.05, 0.05, -0.3,  0.3, -1.0,  0.0 },             // porta nel muro: in basso, dal pavimento a meta' altezza
    { -0.05, 0.05,  0.5,  0.8,  0.4,  0.8 },             // finestra nel muro: in alto
};
// =====================================================================

// Vero se il punto (x, y, z) e' dentro la zona os (bordi compresi)
inline bool dentro(const Zona& os, double x, double y, double z)
{
    return os.x0 <= x && x <= os.x1 && os.y0 <= y && y <= os.y1 && os.z0 <= z && z <= os.z1;
}

// Vero se il punto (x, y, z) e' dentro un ostacolo (i FORI tolgono l'ostacolo)
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& f : FORI)    if (dentro(f, x, y, z)) return false;
    for (const auto& os : OSTACOLI) if (dentro(os, x, y, z)) return true;
    return false;
}

// Maschera della griglia: libero = 1, ostacolo = 0 (resta INF)
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    const int n = N * N * N;
    std::vector<unsigned char> mask(n);
    for (int q = 0; q < n; ++q) {                     // un nodo alla volta
    const int j = q % N, i = (q / N) % N, k = q / (N * N);
    mask[q] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;       // coordinate trasformate in nodi
    }
    // Margine: diventa ostacolo ogni nodo entro MARGINE da un ostacolo
    const int r = (int)std::ceil(MARGINE / h);               // raggio sfera intorno all'ostacolo
    std::vector<unsigned char> margin_mask = mask;
    for (int q = 0; q < n; ++q) {
        if (mask[q]) continue;                              // se ci troviamo nella maschera (quindi in un ostacolo)                                   
    const int k = q / (N * N), i = (q / N) % N, j = q % N;
    const bool bordo = (j > 0 && mask[q - 1]) || (j < N - 1 && mask[q + 1]) ||          // e i nodi toccano un nodo libero
                       (i > 0 && mask[q - N]) || (i < N - 1 && mask[q + N]) ||
                       (k > 0 && mask[q - N * N]) || (k < N - 1 && mask[q + N * N]);
    if (!bordo) continue;                                                               
    for (int dk = -r; dk <= r; ++dk)                          // tutti i nodi entro la sfera di raggio MARGINE
        for (int di = -r; di <= r; ++di)
            for (int dj = -r; dj <= r; ++dj) {
                const int kk = k + dk, ii = i + di, jj = j + dj;
                if (kk < 0 || kk >= N || ii < 0 || ii >= N || jj < 0 || jj >= N) continue;  // non uscire dalla griglia
                if ((dk * dk + di * di + dj * dj) * h * h <= MARGINE * MARGINE)      // dentro la sfera di raggio MARGINE?       
                    margin_mask[(kk * N + ii) * N + jj] = 0;                         // allora il nodo (kk, ii, jj) diventa ostacolo
            }
}
return margin_mask;
}
// Sorgenti della passata 1: i nodi liberi che toccano un ostacolo
inline std::vector<int> nodi_di_bordo(const std::vector<unsigned char>& mask, int N)
{
    std::vector<int> bordo;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                const int q = (k * N + i) * N + j;
                // fa l'opposto di prima
                if (!mask[q]) continue;                                  // solo nodi liberi
                if ((j > 0 && !mask[q - 1]) || (j < N - 1 && !mask[q + 1]) ||
                    (i > 0 && !mask[q - N]) || (i < N - 1 && !mask[q + N]) ||
                    (k > 0 && !mask[q - N * N]) || (k < N - 1 && !mask[q + N * N]))
                    bordo.push_back(q);                                  // che hanno un vicino ostacolo
            }
    return bordo;
}

// FM2: dalla distanza d dagli ostacoli alla lentezza F = 1/v della passata 2
// v = V_MIN + (1 - V_MIN) * min(d / D_SICUREZZA, 1): F = 10 sul bordo, 1 lontano
inline std::vector<float> lentezza_fm2(const std::vector<float>& d)
{
    std::vector<float> F(d.size());
    for (size_t q = 0; q < d.size(); ++q) {
        const double r = std::min((double)d[q] / D_SICUREZZA, 1.0);       // distanza normalizzata: 0 sul bordo, 1 oltre D_SICUREZZA (o se d = INF)
        F[q] = (float)(1.0 / (V_MIN + (1.0 - V_MIN) * r));
    }
    return F;
}
// Controllo iniziale: A e B non devono stare dentro un ostacolo
inline bool ambiente_valido()
{
    return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z);
}

// Dal punto (x, y, z) all'indice del nodo piu' vicino (serve per A e B)
int nodo(int N, double x, double y, double z){
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda
typedef std::pair<float, int> Entry;     // coppia (u, indice del nodo) per la coda di priorita' del FMM

// Risolutore locale 3D (Godunov, Alg. 2 esteso al 3D): dai minimi dei vicini a, b, c al nuovo u
// Prova con 1, 2, poi 3 vicini; usato sia dal FMM (CPU) sia dal Block FIM (GPU)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Nuovo u del nodo (k, i, j) per il FMM: minimo dei vicini per asse + risolutore locale
float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
// Fast Marching Method (Alg. 3) su CPU: il riferimento per il Block FIM
// Fissa i nodi in ordine di tempo crescente con una coda di priorita'
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, const std::vector<int>& srcs, const std::vector<unsigned char>& mask)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    for (int s : srcs){heap.push(Entry(u[s], s)); state[s] = ATTIVO;}

    const int dk[6] = {  0, 0,  0, 0, -1, 1 };       // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1,  0, 0 };       // sinistra, destra, sotto, sopra,
    const int dj[6] = { -1, 1,  0, 0,  0, 0 };       // davanti, dietro

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO || !mask[nb]) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}

In [ ]:
%%writefile fim3d.cu
#include <cstdio>          
#include <cstdlib>         
#include <cuda_runtime.h>
#include <algorithm>       // std::min
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
#include "fmm3d.h"           // Fast Marching
#include "ambiente.h"        // Ambiente

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo
// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Step 1: niter iterazioni; Step 2: 1 iterazione di prova.
__device__ void process_block(int block, int niter, float* u, const float* F,
                             int N, int nbx, float h, const unsigned char* mask, float eps, unsigned char* Cb_out)
{
    // Allocazione statica di shared memory
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];              
    __shared__ bool s_Cn[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi
    // posizione del thread nel suo blocco CUDA: ognuno da 0 a 7
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    // bj: asse x , bi: asse y e bk: asse z; identifica blocco nella griglia
    int bj = block % nbx, bi = (block / nbx) % nbx, bk = block / (nbx * nbx); 
    // numero del thread dentro il blocco; 
    int tid = (tz*BLOCK + ty) * BLOCK + tx; // 0, ..., 511
    // posizione globale rispetto x,y e z: lungo tutta la griglia
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;
    int idx = (k * N + i) * N + j;  // 0, ..., N^3-1
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // nodo da non aggiornare: fuori dalla griglia o maschera
    float f = inside ? F[idx] : 1.0f;
    // Global -> Shared
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        // posizione nella shared memory
        int sz = m / ((BLOCK + 2) * (BLOCK + 2)), sy = (m / (BLOCK + 2)) % (BLOCK + 2), sx = m % (BLOCK + 2);
        // coordinate globali della cella (sz, sy, sx): -1 perché c'è l'halo                          
        int gi = bi * BLOCK + sy - 1, gj = bj * BLOCK + sx - 1, gk = bk * BLOCK + sz - 1;
        
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();
    bool conv = true;
    for (int it = 0; it < niter; ++it){
        // p: valore del nodo all'inizio di questa iterazione
        float p = s_u[tz+1][ty+1][tx+1];
        // q: nuovo valore calcolato in questa iterazione
        float q;
        if (fixed) {q = p;}
        else{
        // tx, ty, tz: 0,...,7; +1: nella shared memory c'è l'halo
        q = solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        }    
        __syncthreads();
        s_u[tz+1][ty + 1][tx + 1] = q;
        conv = fabsf(p - q) < eps;
        if (!__syncthreads_or(!conv)) break;   // (B) tutti hanno scritto; se nessun nodo e' cambiato, esci
    }
    // Shared -> Global: ogni thread riscrive il proprio nodo
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];
    // Cn del nodo: vero se non è cambiato nell'ultima iterazione
    s_Cn[tid] = conv;
    __syncthreads();
    for (int reduct = BLOCK * BLOCK * BLOCK / 2; reduct > 0; reduct /= 2) {     // 256, ..., 4, 2, 1
        if (tid < reduct) s_Cn[tid] = s_Cn[tid] && s_Cn[tid + reduct];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_Cn[0];
}
// Step1: ogni blocco della lista A fa niter iterazioni
__global__ void update_active_blocks(const int* active_list, float* u, const float* F,
                                    int N, int nbx, float h, const unsigned char* mask, int niter,
                                    float eps, unsigned char* Cb_step1)
{
    // blockIdx è il numero del blocco CUDA nel lancio in corso
    process_block(active_list[blockIdx.x], niter, u, F, N, nbx, h, mask, eps, Cb_step1);
    // (N = 33) giro: 1 -> blocchi CUDA lanciati: 1 -> blockIdx.x: 0 -> block = active_list[blockIdx.x]: 62
    // (N = 33) giro: 3 -> blocchi CUDA lanciati: 6 -> blockIdx.x: 0, 1, ..., 5 -> block = active_list[blockIdx.x]: 61, 63, 57, 67, 37, 87
}

// costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* in_list,
                                  int* active_list, int* n_active, int n_blocks) 
{
    // blockIdx: blocco nel lancio in corso; blockDim (monodimensionale CUDA): 256; threadIdx:0,...,255
    int b = blockIdx.x * blockDim.x + threadIdx.x;           // numero del thread(blocco) su tutto il lancio
    // N = 65 -> nbx = 9 -> n_blocks = 729 -> list_blocks = 3 -> thread lanciati = 768 -> thread in eccesso = 39 
    if (b >= n_blocks) return;                               // thread(blocco) in eccesso lanciati da CUDA                                   
    in_list[b] = !Cb[b];                                      // fotografia della lista A di questo giro
    if (!Cb[b]) active_list[atomicAdd(n_active, 1)] = b;               // se NON convergente aggiungi thread(blocco) alla lista A
}
// Step 2: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* active_list, int n_active, int nbx, const unsigned char* in_list,
                                     const unsigned char* Cb_step1, unsigned char* Cb,
                                     int* is_candidate, int* cand_list, int* cand_count)
{
    int pos = blockIdx.x * blockDim.x + threadIdx.x;
    // se è maggiore o uguale al numero di blocchi attivi fai return
    if (pos >= n_active) return;                       
    int b = active_list[pos];         // un blocco della lista A; N = 33 -> A ={61, 63, ...,87}; active_list[0] =61
    Cb[b] = Cb_step1[b];             // leggi il valore di Cb
    if (!Cb_step1[b]) return;        // proseguono solo i blocchi appena convergenti
    // bj: blocco lungo x; bi: blocco lungo y; bk: blocco lungo z
    int bj = b % nbx, bi = (b / nbx) % nbx, bk = b / (nbx * nbx);
    // sinistra, destra, sotto, sopra e davanti, dietro
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};
    // verifico se si trova nella griglia
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // scarta blocchi fuori dalla griglia
        // scarta blocchi attivi e non convergenti
        if (in_list[nb[d]] && !Cb_step1[nb[d]]) continue;          // d: indica la direzione; N = 33; nb[d] = [61, 63, 57, 67, 37, 87];      
        if (atomicExch(&is_candidate[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}
// Step 2: ogni candidato fa 1 iterazione; se cambia, diventa attivo
__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* is_candidate,
                                        float* u, const float* F, int N, int nbx, float h,
                                        const unsigned char* mask, float eps, unsigned char* Cb)
{
    // scarta blocchi CUDA in eccesso
    if (blockIdx.x >= *cand_count) return;
    int b = cand_list[blockIdx.x];
    // blocchi candidati: niter = 1 -> con la nuova informazione dal blocco convergente, questo vicino cambia?
    process_block(b, 1, u, F, N, nbx, h, mask, eps, Cb);
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) is_candidate[b] = 0;   // pronto per il giro successivo
}
// Memoria della GPU: si alloca alla prima chiamata e si riusa finche' il main non chiama libera_gpu()
static bool allocata = false;
static float *d_u, *d_F; static unsigned char *d_Cb, *d_Cb_step1, *d_in_list, *d_mask;
static int *d_active_list, *d_n_active, *d_is_candidate, *d_cand_list, *d_cand_count;
void libera_gpu()                                          // libera la memoria e rimette allocata = false
{
    if (!allocata) return;
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F)); CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb_step1)); CUDA_CHECK(cudaFree(d_mask));
    CUDA_CHECK(cudaFree(d_in_list)); CUDA_CHECK(cudaFree(d_active_list)); CUDA_CHECK(cudaFree(d_n_active));
    CUDA_CHECK(cudaFree(d_is_candidate)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    allocata = false;
}
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, int niter, float* ms)
{
    // nbx: numero di blocchi per lato; n_blocks: blocchi totali
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx;
    // bytes: dimensione totale griglia 3D
    const size_t bytes = (size_t)N * N * N * sizeof(float);
    // Cb: si inizializza vettore di n_blocks valori a 1
    std::vector<unsigned char> Cb(n_blocks, 1);
    // strato, riga e colonna del nodo sorgente
    for (int s : srcs){
    const int sk = s / (N * N), si = (s / N) % N, sj = s % N;
    // Cb: vale 1 tranne nel blocco della sorgente (0) 
    Cb[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;
    }

    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    if (!allocata) {
    CUDA_CHECK(cudaMalloc(&d_u, bytes)); CUDA_CHECK(cudaMalloc(&d_F, bytes)); CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb_step1, n_blocks)); CUDA_CHECK(cudaMalloc(&d_in_list, n_blocks)); CUDA_CHECK(cudaMalloc(&d_active_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_n_active, sizeof(int))); CUDA_CHECK(cudaMalloc(&d_mask, N*N*N));
    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_is_candidate, n_blocks * sizeof(int))); CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));
    allocata = true;
    }
    // si copiano d_u, d_F e d_Cb dalla CPU alla GPU
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice)); CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb.data(), n_blocks, cudaMemcpyHostToDevice)); CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), N*N*N, cudaMemcpyHostToDevice));
    // si azzerano d_Cb_step1 e d_is_candidate
    CUDA_CHECK(cudaMemset(d_Cb_step1, 0, n_blocks)); CUDA_CHECK(cudaMemset(d_is_candidate, 0, n_blocks * sizeof(int)));
    // list_blocks: blocchi CUDA da 256 threads per avere un thread per ogni blocco di nodi 
    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;
    // si misura l'ora prima del ciclo
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start)); CUDA_CHECK(cudaEventCreate(&stop)); CUDA_CHECK(cudaEventRecord(start));                     
    int iter = 0;
    while (true) {
        // azzera il numero di blocchi attivi                                           
        int n_active = 0;  // contatore su CPU
        CUDA_CHECK(cudaMemset(d_n_active, 0, sizeof(int)));    // contatore su GPU
        // lancia list_blocks CUDA da 256 threads ciascuno, sufficiente per avere un thread ogni blocco
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_in_list, d_active_list, d_n_active, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        // aggiorna n_active con d_n_active
        CUDA_CHECK(cudaMemcpy(&n_active, d_n_active, sizeof(int), cudaMemcpyDeviceToHost));
        if (n_active == 0) break;  // se n_active == 0, lista A vuota, termina ciclo
        ++iter;                 // conta i giri
        // Step 1: foreach b in A in parallelo; lancia n_active blocchi 8x8x8
        update_active_blocks<<<n_active, dim3(BLOCK, BLOCK, BLOCK)>>>(d_active_list, d_u, d_F, N, nbx, h, d_mask, niter, eps, d_Cb_step1);    // si lanciano n_active blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        // azzeriamo il contatore dei candidati sulla GPU
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        // lancia n_active blocchi CUDA da 256 threads
        build_candidate_list<<<(n_active + threads-1) / threads, threads>>>(d_active_list, n_active, nbx, d_in_list, d_Cb_step1, d_Cb,
                                                                          d_is_candidate, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        // ogni blocco ha al più 6 vicini, quindi 6*n_active
        update_candidate_blocks<<<std::min(6 * n_active, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_is_candidate,
                                                                                    d_u, d_F, N, nbx, h, d_mask, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    return iter;
}
int main()
{
    // Informazioni sulla GPU usata
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = 8;                                     // iterazioni locali massime (uscita anticipata): la migliore con gli ostacoli (prove 12 / 24)

    if (!ambiente_valido()) {                                // A o B dentro un ostacolo: il problema non ha senso
        printf("Errore: A o B sono dentro un ostacolo, correggi ambiente.h\n");
        return 1;
    }
    printf("    N   giri   max|FIM - FMM|   u(A) FIM   u(A) FMM   t FMM [ms]   t FIM [ms]   speedup\n");

    for (int Nk : { 33, 65, 129, 193, 257, 385 }) {
        const int ripetizioni = 3;                           // ogni tempo = il minimo di 3 misure...
        const float hk = 2.0f / (Nk - 1);                    // passo della griglia nel cubo[-1, 1]^3
        const int n = Nk * Nk * Nk;                          // numero di nodi
        const std::vector<unsigned char> mask = costruisci_maschera(Nk);    // 1 = libero, 0= ostacolo
        const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };          // la sorgente e' l'arrivo B
        const int iA = nodo(Nk, A.x, A.y, A.z);              // nodo della partenza A

        // passata 1: distanza dagli ostacoli (F = 1), con FIM e FMM
        const std::vector<int> bordo = nodi_di_bordo(mask, Nk);
        std::vector<float> d0(n, INF);                       // condizione iniziale: 0 sulbordo, INF altrove
        for (int s : bordo) d0[s] = 0.0f;
        const std::vector<float> uno(n, 1.0f);               // F = 1: u = distanza
        std::vector<float> d_ost, d_fmm;
        double t1_fmm = 1e30, t1_fim = 1e30;
        for (int r = 0; r < ripetizioni; ++r) {              // 3 misure, si tiene la piu'veloce
            // calcolo del fmm
            d_fmm = d0;
            auto t0 = std::chrono::high_resolution_clock::now();
            fmm(d_fmm, uno, Nk, hk, bordo, mask);            // CPU: passata 1 con il FMM
            auto t1 = std::chrono::high_resolution_clock::now();
            t1_fmm = std::min(t1_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());
            // calcolo del fim
            d_ost = d0;
            float ms = 0.0f;
            block_fim(d_ost, uno, Nk, hk, bordo, mask, eps, niter, &ms);   // GPU: passata 1 con il FIM
            t1_fim = std::min(t1_fim, (double)ms);           // solo i giri (eventi CUDA), come nel codice di Jeong
        }
        float diff1 = 0.0f;                                  // la distanza del FIM devecoincidere con quella del FMM
        for (int q = 0; q < n; ++q) diff1 = fmaxf(diff1, fabsf(d_ost[q] - d_fmm[q]));

        // FM2: lentezza F dalla distanza dagli ostacoli
        const std::vector<float> Fk = lentezza_fm2(d_ost);
        // passata 2: tempo di arrivo da B con questa F, con FMM e FIM
        std::vector<float> u0(n, INF);                       // condizione iniziale: 0 in B, INF altrove
        for (int s : srcs) u0[s] = 0.0f;
        std::vector<float> uk_fmm, uk_fim;
        double t_fmm = 1e30, t_fim = 1e30;
        int it = 0;
        for (int r = 0; r < ripetizioni; ++r) {
            // calcolo del fmm
            uk_fmm = u0;                                     // ogni misura riparte da capo
            auto t0 = std::chrono::high_resolution_clock::now();
            fmm(uk_fmm, Fk, Nk, hk, srcs, mask);             // CPU: passata 2 con il FMM (Alg. 3)
            auto t1 = std::chrono::high_resolution_clock::now();
            t_fmm = std::min(t_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());
            // calcolo del fim
            uk_fim = u0;
            float ms = 0.0f;
            it = block_fim(uk_fim, Fk, Nk, hk, srcs, mask, eps, niter, &ms);   // GPU:passata 2 con il Block FIM (Alg. 7)
            t_fim = std::min(t_fim, (double)ms);             // solo i giri (eventi CUDA), come nel codice di Jeong
        }

        // Controlli: FIM = FMM, nessun NaN, ostacoli a INF
        float diff = diff1;
        int n_nan = 0, ost_non_inf = 0;
        for (int q = 0; q < n; ++q) {
            diff = fmaxf(diff, fabsf(uk_fim[q] - uk_fmm[q]));
            if (std::isnan(uk_fim[q])) ++n_nan;
            if (!mask[q] && uk_fim[q] < INF) ++ost_non_inf;
        }
        if (n_nan > 0 || ost_non_inf > 0)                    // si stampa solo se qualcosa non va
            printf("ATTENZIONE (N = %d): %d valori NaN, %d nodi ostacolo raggiuntidall'onda\n", Nk, n_nan, ost_non_inf);

        const double t_fmm_tot = t1_fmm + t_fmm, t_fim_tot = t1_fim + t_fim;   // FM2 completo
        if (uk_fim[iA] >= INF)                               // l'onda partita da B non e' mai arrivata ad A
            printf("%5d   %4d   %14.2e   A NON RAGGIUNGIBILE da B\n", Nk, it, diff);
        else
            printf("%5d   %4d   %14.2e   %8.4f   %8.4f   %10.1f   %10.2f   %7.1f\n",
                   Nk, it, diff, uk_fim[iA], uk_fmm[iA], t_fmm_tot, t_fim_tot, t_fmm_tot / t_fim_tot);

        if (Nk == 257) {                                      // per i grafici (Python): soluzione, maschera e ambiente
            std::ofstream out("fim_257_ostacoli.bin", std::ios::binary);
            out.write(reinterpret_cast<const char*>(uk_fim.data()), uk_fim.size() * sizeof(float));
            std::ofstream om("mask_257.bin", std::ios::binary);
            om.write(reinterpret_cast<const char*>(mask.data()), mask.size());
            std::ofstream amb("ambiente.txt");               // l'ambiente di ambiente.h, una riga per elemento
            amb.precision(15);                               // 15 cifre: i numeri arrivano a Python esattamente come in ambiente.h
            amb << "MARGINE " << MARGINE << "\n";
            amb << "A " << A.x << " " << A.y << " " << A.z << "\n";
            amb << "B " << B.x << " " << B.y << " " << B.z << "\n";
            for (const auto& os : OSTACOLI) amb << "ZONA " << os.x0 << " " << os.x1 << " " << os.y0 << " " << os.y1 << " " << os.z0 << " " << os.z1 << "\n";
            for (const auto& f : FORI)    amb << "FORO " << f.x0 << " " << f.x1 << " " << f.y0 << " " << f.y1 << " " << f.z0 << " " << f.z1 << "\n";
            std::ofstream of("F_257.bin", std::ios::binary);  // la lentezza F usata nellapassata 2 (serve a Python)
            of.write(reinterpret_cast<const char*>(Fk.data()), Fk.size() * sizeof(float));
        }
        libera_gpu();
    }

    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 fim3d.cu -o fim3d && ./fim3d

In [ ]:
# PERCORSO DEL DRONE nella stanza (N = 257): discesa lungo u da A a B, verifica e vista 3D con il fronte d'onda
# Legge i file scritti da fim3d.cu: fim_257_ostacoli.bin, mask_257.bin, F_257.bin, ambiente.txt

import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import RegularGridInterpolator
from scipy.spatial import cKDTree
from skimage.measure import marching_cubes

# 1) DATI scritti dal C++ (N = 257): u, maschera, F, ambiente
N = 257
h = 2.0 / (N - 1)
u = np.fromfile("fim_257_ostacoli.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)   # u[k, i, j] = u(z, y, x)
mask = np.fromfile("mask_257.bin", dtype=np.uint8).reshape(N, N, N)                             # 1 = libero, 0 = ostacolo
libero = mask == 1
x = -1.0 + h * np.arange(N)
Fl = np.fromfile("F_257.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)              # lentezza F usata dal FIM

amb = {"ZONA": [], "FORO": []}
for riga in open("ambiente.txt"):
    nome, *valori = riga.split()
    valori = [float(v) for v in valori]
    if nome == "A":         A = np.array(valori)
    elif nome == "B":       B = np.array(valori)
    elif nome == "MARGINE": MARGINE = valori[0]
    else:                   amb[nome].append(tuple(valori))
OSTACOLI, FORI = amb["ZONA"], amb["FORO"]

def nodo(p):                                                        # nodo (k, i, j) piu' vicino al punto p = (x, y, z)
    return tuple(np.round((np.asarray(p)[::-1] + 1.0) / h).astype(int))
def coord(n):                                                       # coordinate (x, y, z) del nodo n = (k, i, j)
    return -1.0 + h * np.array(n[::-1], dtype=float)

if u[nodo(A)] >= 1e29:                                              # u(A) = INF: l'onda partita da B non e' mai arrivata ad A
    raise SystemExit("A non e' raggiungibile da B: nessun percorso (controlla ostacoli e fori in ambiente.h)")


# 2) PERCORSO da A a B (il piu' veloce secondo u)
# Discesa: da A si passa al vicino libero (26) da cui si arriva prima a B
mosse = [(dk, di, dj) for dk in (-1, 0, 1) for di in (-1, 0, 1) for dj in (-1, 0, 1) if (dk, di, dj) != (0, 0, 0)]
n, nB = nodo(A), nodo(B)
nodi = [n]
while n != nB:
    vicini = [(n[0] + a, n[1] + b, n[2] + c) for a, b, c in mosse]
    vicini = [m for m in vicini if all(0 <= t < N for t in m) and libero[m]]
    n = min(vicini, key=lambda m: u[m] + 0.5 * (Fl[n] + Fl[m]) * h * np.linalg.norm(np.subtract(m, n)))   # u del vicino + tempo per arrivarci
    nodi.append(n)

# Accorciamento: si salta al nodo piu' avanti visibile senza ostacoli
# e che non fa perdere tempo
peso = RegularGridInterpolator((x, x, x), libero.astype(float))     # = 1 solo se gli 8 nodi attorno sono liberi
F_interp = RegularGridInterpolator((x, x, x), Fl)
def tempo(p, q):                                                    # tempo lungo il segmento p-q: somma di F * ds
    t = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    medi = p + (t[:-1] + t[1:]) / 2 * (q - p)
    return np.sum(F_interp(medi[:, ::-1])) * np.linalg.norm(q - p) / (len(t) - 1)
def si_vede(p, q):                                                  # vero se il segmento p-q non attraversa ostacoli (maschera)
    t = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    return np.all(peso((p + t * (q - p))[:, ::-1]) > 0.999)
P = [coord(m) for m in nodi]
t_nodi = np.concatenate([[0], np.cumsum([tempo(P[i], P[i + 1]) for i in range(len(P) - 1)])])   # tempo lungo la discesa
percorso, a = [P[0]], 0
while a < len(P) - 1:
    b = len(P) - 1
    while b > a + 1 and not (si_vede(P[a], P[b]) and tempo(P[a], P[b]) <= t_nodi[b] - t_nodi[a]):
        b -= 1
    percorso.append(P[b])
    a = b
percorso = np.array(percorso)
lunghezza = np.sum(np.linalg.norm(np.diff(percorso, axis=0), axis=1))
tempo_percorso = sum(tempo(percorso[i], percorso[i + 1]) for i in range(len(percorso) - 1))


# 3) VERIFICA con la geometria esatta degli ostacoli

def dentro(b, p):
    return (b[0] <= p[:, 0]) & (p[:, 0] <= b[1]) & (b[2] <= p[:, 1]) & (p[:, 1] <= b[3]) & (b[4] <= p[:, 2]) & (p[:, 2] <= b[5])
def ostacolo_vero(p):                                               # vero per i punti p (righe x, y, z) dentro un ostacolo
    occ = np.zeros(len(p), bool)
    for s in OSTACOLI: occ |= dentro(s, p)
    for f in FORI:    occ &= ~dentro(f, p)
    return occ

fitto = np.concatenate([p + np.linspace(0, 1, 200, endpoint=False)[:, None] * (q - p)
                        for p, q in zip(percorso[:-1], percorso[1:])] + [percorso[-1:]])   # 200 punti per segmento
g = np.linspace(-1, 1, 201)                                         # griglia fine (passo 0.01) per la distanza dagli ostacoli
G = np.stack(np.meshgrid(g, g, g, indexing="ij")[::-1], axis=-1).reshape(-1, 3)
dist, _ = cKDTree(G[ostacolo_vero(G)]).query(fitto)

print(f"u(A) = {u[nodo(A)]:.4f}   tempo lungo il percorso = {tempo_percorso:.4f}   (devono quasi coincidere)")
print(f"lunghezza del percorso = {lunghezza:.4f}   linea retta |A - B| = {np.linalg.norm(A - B):.4f}")
print(f"punti del percorso controllati: {len(fitto)}, dentro un ostacolo: {ostacolo_vero(fitto).sum()}")
print(f"distanza minima del percorso dagli ostacoli: {dist.min():.3f}  (MARGINE = {MARGINE})")
for i, f in enumerate(FORI):
    sel = dentro(f, fitto)
    print(f"foro {i + 1}: " + (f"attraversato in ({fitto[sel, 0].mean():.2f}, {fitto[sel, 1].mean():.2f}, {fitto[sel, 2].mean():.2f})"
                               if sel.any() else "non usato"))

# 4) VISTA 3D: ostacoli, percorso e fronte d'onda (slider)
SV = (N - 1) // 128                                                 # superfici 3D su un nodo ogni SV (2 a N = 257): grafico leggero
xv, hv = x[::SV], h * SV
Pn = np.stack(np.meshgrid(xv, xv, xv, indexing="ij")[::-1], axis=-1).reshape(-1, 3)
pieno = ostacolo_vero(Pn).reshape(len(xv), len(xv), len(xv))       # ostacoli veri sulla griglia del grafico
vo, fo, _, _ = marching_cubes(np.pad(pieno.astype(float), 1), level=0.5, spacing=(hv, hv, hv))
po = vo - hv - 1.0                                                  # vertici in ordine (z, y, x)
ostacoli = go.Mesh3d(x=po[:, 2], y=po[:, 1], z=po[:, 0], i=fo[:, 0], j=fo[:, 1], k=fo[:, 2],
                     color="lightgray", opacity=0.35, name="ostacoli", flatshading=True)
s_perc = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(percorso, axis=0), axis=1))])
linea = go.Scatter3d(x=percorso[:, 0], y=percorso[:, 1], z=percorso[:, 2], mode="lines+markers", name="percorso",
                     line=dict(color=s_perc[-1] - s_perc, colorscale="Reds_r", width=9), marker=dict(size=2, color="darkred"))
punti = go.Scatter3d(x=[A[0], B[0]], y=[A[1], B[1]], z=[A[2], B[2]], mode="markers+text", text=["A", "B"],
                     marker=dict(size=7, color=["green", "blue"]), textfont=dict(size=16), name="A e B")

w = np.where(libero, u, u[libero].max() * 1.05)[::SV, ::SV, ::SV]  # negli ostacoli un valore alto ma finito (griglia del grafico)
istanti = np.linspace(u[libero].max() * 0.04, u[libero].max() * 0.98, 15)
def fronte(t):                                                      # superficie u = t (fronte all'istante t)
    v, f, _, _ = marching_cubes(w, level=t, spacing=(hv, hv, hv), step_size=2)
    p = v - 1.0
    return go.Mesh3d(x=p[:, 2], y=p[:, 1], z=p[:, 0], i=f[:, 0], j=f[:, 1], k=f[:, 2],
                     color="deepskyblue", opacity=0.45, name="fronte u = t")

fig = go.Figure(data=[ostacoli, linea, punti, fronte(istanti[0])],
                frames=[go.Frame(data=[fronte(t)], traces=[3], name=f"{t:.2f}") for t in istanti])
fig.update_layout(
    title="Ostacoli, percorso da A a B e fronte d'onda partito da B",
    scene=dict(xaxis=dict(range=[-1, 1]), yaxis=dict(range=[-1, 1]), zaxis=dict(range=[-1, 1]), aspectmode="cube"),
    height=750, margin=dict(l=0, r=0, t=40, b=0),
    sliders=[dict(currentvalue=dict(prefix="fronte u = "), pad=dict(t=30),
                  steps=[dict(label=f"{t:.2f}", method="animate",
                              args=[[f"{t:.2f}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for t in istanti])])
fig.write_html("percorso_3d.html", include_plotlyjs="cdn")         # anche come file da scaricare: si apre nel browser
fig.show()

In [ ]:
# ANIMAZIONE: il drone va da A a B secondo il tempo (usa la cella precedente)
# Frame a tempo costante: vicino agli ostacoli il drone rallenta
n_frame = 60
s_fitto = np.linspace(0.0, s_perc[-1], 2000)                        # 2000 punti equidistanti lungo il percorso
P_fitto = np.stack([np.interp(s_fitto, s_perc, percorso[:, c]) for c in range(3)], axis=1)
medi = (P_fitto[:-1] + P_fitto[1:]) / 2                             # punto medio di ogni trattino
dt = F_interp(medi[:, ::-1]) * np.diff(s_fitto)                     # tempo di ogni trattino: F * ds
t_fitto = np.concatenate([[0.0], np.cumsum(dt)])                    # tempo per arrivare a ogni punto
t_tot = t_fitto[-1]                                                 # tempo totale (circa u(A))
istanti_t = np.linspace(0.0, t_tot, n_frame)                        # istanti equidistanti nel tempo
passi = np.searchsorted(t_fitto, istanti_t).clip(0, len(t_fitto) - 1)   # punto raggiunto in ogni istante

def tratto(m):                                                      # linea da A fino al punto m e drone sul punto m
    return [go.Scatter3d(x=P_fitto[:m + 1, 0], y=P_fitto[:m + 1, 1], z=P_fitto[:m + 1, 2], mode="lines",
                         line=dict(color="darkred", width=9), name="percorso"),
            go.Scatter3d(x=[P_fitto[m, 0]], y=[P_fitto[m, 1]], z=[P_fitto[m, 2]], mode="markers",
                         marker=dict(size=6, color="orange"), name="drone")]

anim = go.Figure(data=[ostacoli, punti] + tratto(0),
                 frames=[go.Frame(data=tratto(m), traces=[2, 3], name=str(f)) for f, m in enumerate(passi)])
anim.update_layout(
    title=f"Il drone va da A a B secondo il tempo (tempo totale {t_tot:.2f}): rallenta vicino agli ostacoli",
    scene=dict(xaxis=dict(range=[-1, 1]), yaxis=dict(range=[-1, 1]), zaxis=dict(range=[-1, 1]), aspectmode="cube"),
    height=750, margin=dict(l=0, r=0, t=40, b=80),
    updatemenus=[dict(type="buttons", direction="left", x=0.1, y=0, xanchor="right", yanchor="top", pad=dict(r=10, t=70), buttons=[
        dict(label="Play", method="animate",
             args=[None, dict(frame=dict(duration=80, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label="Pausa", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
    sliders=[dict(x=0.1, len=0.9, y=0, yanchor="top", currentvalue=dict(prefix="tempo t = "), pad=dict(t=50),
                  steps=[dict(label=f"{t:.1f}", method="animate",
                              args=[[str(f)], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for f, t in enumerate(istanti_t)])])
anim.write_html("percorso_animato.html", include_plotlyjs="cdn")    # anche come file da scaricare: si apre nel browser
anim.show()